# Feature Engineering
To bridge the gap between raw data and usable data input. Stationary features created should value add to the complex deep learning model. The aim of this notebook is to build a baseline model that can contrast to future deep learning models for comparison purpose.

## A: Data Cleaning & Transformation

Raw stock prices are non-stationary, as they tend to wander. This may pose confusion to our model, thus transforming them makes the data more stable and additive. 

One good way to do it is through Log Returns. I took reference from this article: [How to Analyze Financial Returns in Python](https://www.kaggle.com/code/meiliaa/how-to-analyze-financial-returns-in-python). 
* **Simple returns**: Intuitive interpretation ("stock gained 5%"), used for portfolio aggregation
* **Log returns**: Time-additive property is better for modeling

### A.1 Load Data from TimescaleDB

Reuse the same connection pattern as `01_exploratory_data_analysis.ipynb`. No need to re-fetch from an external API or URL — the data already lives in the `market_data` table from the EDA step.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

db_url = (
    f"postgresql://{DB_USER}:"
    f"{DB_PASSWORD}@"
    f"{DB_HOST}:"
    f"{DB_PORT}/"
    f"{DB_NAME}"
)
engine = create_engine(db_url)

df = pd.read_sql("SELECT * FROM market_data ORDER BY time ASC", engine)
df["time"] = pd.to_datetime(df["time"])
df.set_index("time", inplace=True)
print("Data loaded successfully.")

print(f"Rows loaded: {len(df)}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

Data loaded successfully.
Rows loaded: 2626
Date range: 2016-01-04 00:00:00+00:00 to 2026-06-12 00:00:00+00:00


,symbol,price_close,volume
time,,,
2016-01-04 00:00:00+00:00,NVDA,0.788626,358076000
2016-01-05 00:00:00+00:00,NVDA,0.801295,490272000
2016-01-06 00:00:00+00:00,NVDA,0.768161,449344000
2016-01-07 00:00:00+00:00,NVDA,0.737708,645304000
2016-01-08 00:00:00+00:00,NVDA,0.721872,398472000


### A.2 Compute Log Returns

Note: the EDA notebook from Week 1 used `pct_change()`, which gives **simple** returns. Here we compute **log returns** instead, since they're time-additive and tend to be more stationary - better suited for modeling.

$$r_t = \ln(P_t) - \ln(P_{t-1})$$

We use `numpy` to help us with this array-wise calculation.

In [3]:
df["log_returns"] = np.log(df["price_close"]) - np.log(df["price_close"].shift(1))

# Drop the first row (NaN from the shift)
df = df.dropna(subset=["log_returns"]).copy()

df[["price_close", "log_returns"]].head()

,price_close,log_returns
time,,
2016-01-05 00:00:00+00:00,0.801295,0.015937
2016-01-06 00:00:00+00:00,0.768161,-0.042229
2016-01-07 00:00:00+00:00,0.737708,-0.040452
2016-01-08 00:00:00+00:00,0.721872,-0.021700
2016-01-11 00:00:00+00:00,0.723090,0.001685
